# Simple CNN and Laplacian Image Attack

This standalone tutorial notebook implements a small convolutional neural network without PyTorch or TensorFlow, trains it on handwritten digit images, and then studies the distortion

$$
\widehat A = A + \nabla^2 A.
$$

The notebook is intended for a tutorial following an introductory lecture on fully connected neural networks and convolutional neural networks.

## Learning objectives

By the end of the notebook, students should be able to:

1. Represent a grayscale image as a matrix of pixel intensities.
2. Explain the basic structure of a convolutional neural network.
3. Train a small CNN classifier.
4. Derive the five-point discrete Laplacian.
5. Implement the distortion

$$
\widehat A = A + \nabla^2 A.
$$

6. Compare CNN predictions before and after the distortion.
7. Measure test accuracy as a function of distortion strength.

The executed demonstration uses the built-in `scikit-learn` handwritten digits dataset. This keeps the notebook lightweight, avoids external downloads, and avoids deep-learning installation issues. A final section gives clear instructions for students who want to repeat the same experiment with MNIST.

## 1. Imports

We use only standard scientific Python packages:

- `numpy` for numerical arrays,
- `matplotlib` for plotting,
- `scikit-learn` for the built-in handwritten digits dataset and train-test splitting.

This notebook deliberately avoids `torch` and `tensorflow` so that it can run in a standard scientific Python environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(0)

print("NumPy version:", np.__version__)

## 2. Images as matrices

A grayscale image can be represented as a matrix

$$
A =
\begin{pmatrix}
A_{1,1} & A_{1,2} & \cdots & A_{1,W}\\
A_{2,1} & A_{2,2} & \cdots & A_{2,W}\\
\vdots & \vdots & \ddots & \vdots\\
A_{H,1} & A_{H,2} & \cdots & A_{H,W}
\end{pmatrix}.
$$

Each entry is a pixel intensity. In this notebook, pixel values are normalized so that

$$
0 \leq A_{i,j} \leq 1.
$$

The built-in digits dataset contains grayscale handwritten digit images. Each image is small, with spatial size 8 by 8, which makes the full NumPy implementation fast enough for a tutorial.

In [ ]:
digits = load_digits()

X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

print("Original image array shape:", X.shape)
print("Label array shape:", y.shape)
print("Pixel range:", X.min(), "to", X.max())

# Add a channel dimension.
# Shape changes from (N, H, W) to (N, C, H, W), where C=1 for grayscale.
X = X[:, np.newaxis, :, :]

print("Image array shape after adding channel dimension:", X.shape)

In [ ]:
plt.figure(figsize=(10, 3))

for k in range(10):
    plt.subplot(2, 5, k + 1)
    plt.imshow(X[k, 0], cmap="gray", vmin=0, vmax=1)
    plt.title(f"label = {y[k]}")
    plt.axis("off")

plt.suptitle("Examples from the handwritten digits dataset")
plt.tight_layout()
plt.show()

## 3. Train, validation, and test split

We split the dataset into three parts:

- the training set is used to update the network parameters,
- the validation set is used to monitor learning,
- the test set is kept aside for final evaluation.

The test set is also used later to measure how strongly the Laplacian distortion reduces accuracy.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=0,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=1,
    stratify=y_train_full,
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

## 4. The CNN architecture

We use a deliberately small convolutional neural network:

$$
\text{image}
\longrightarrow
\text{convolution}
\longrightarrow
\text{ReLU}
\longrightarrow
\text{max pooling}
\longrightarrow
\text{dense classifier}
\longrightarrow
\text{softmax}.
$$

For the 8 by 8 digit images, the shape transformation is

$$
8\times 8\times 1
\longrightarrow
8\times 8\times 8
\longrightarrow
4\times 4\times 8
\longrightarrow
10.
$$

The final output contains ten class probabilities, one for each digit.

### Convolution

For one 3 by 3 filter, the convolutional response at pixel position $$(i,j)$$ is

$$
Z_{i,j}
=
b+
\sum_{r=-1}^{1}
\sum_{s=-1}^{1}
K_{r,s} A_{i+r,j+s}.
$$

In neural-network software this operation is often implemented as cross-correlation rather than as a flipped mathematical convolution. That distinction is not important for this tutorial because the filter entries are learned.

### ReLU

The ReLU activation is

$$
\operatorname{ReLU}(z)=\max\{0,z\}.
$$

### Max pooling

A 2 by 2 max-pooling operation maps

$$
\begin{pmatrix}
a & b\\
c & d
\end{pmatrix}
\longmapsto
\max\{a,b,c,d\}.
$$

This reduces the spatial resolution by a factor of two.

## 5. CNN implementation in NumPy

The next cell implements a minimal CNN directly in NumPy. It includes:

- a convolutional layer,
- ReLU,
- 2 by 2 max pooling,
- a dense classifier,
- softmax and cross-entropy loss,
- backpropagation,
- momentum stochastic gradient descent.

This implementation is intentionally compact and pedagogical. For real projects, one would normally use a deep-learning library. Here the network is implemented explicitly so that the notebook remains independent of PyTorch and TensorFlow.

In [ ]:
class TinyNumPyCNN:
    # Minimal CNN implemented directly in NumPy.
    # Architecture:
    # input -> conv 3x3 -> ReLU -> 2x2 maxpool -> dense -> softmax.

    def __init__(self, image_shape=(8, 8), n_filters=8, n_classes=10, seed=42):
        self.image_height, self.image_width = image_shape
        self.n_filters = n_filters
        self.n_classes = n_classes

        if self.image_height % 2 != 0 or self.image_width % 2 != 0:
            raise ValueError("This simple demo assumes even image height and width.")

        pooled_size = n_filters * (self.image_height // 2) * (self.image_width // 2)

        rng = np.random.default_rng(seed)

        # Convolution weights: (filters, channels, kernel height, kernel width)
        self.Wc = 0.5 * rng.normal(
            0.0,
            np.sqrt(2.0 / 9.0),
            size=(n_filters, 1, 3, 3),
        ).astype(np.float32)

        self.bc = np.zeros(n_filters, dtype=np.float32)

        # Dense layer weights
        self.Wd = 0.5 * rng.normal(
            0.0,
            np.sqrt(2.0 / pooled_size),
            size=(pooled_size, n_classes),
        ).astype(np.float32)

        self.bd = np.zeros(n_classes, dtype=np.float32)

        self.velocity = {
            "Wc": np.zeros_like(self.Wc),
            "bc": np.zeros_like(self.bc),
            "Wd": np.zeros_like(self.Wd),
            "bd": np.zeros_like(self.bd),
        }

    def parameters(self):
        return {
            "Wc": self.Wc,
            "bc": self.bc,
            "Wd": self.Wd,
            "bd": self.bd,
        }

    def forward(self, X):
        B, C, H, W = X.shape

        if C != 1:
            raise ValueError("This tutorial CNN expects grayscale images with one channel.")

        if H != self.image_height or W != self.image_width:
            raise ValueError(
                f"Expected images of shape {(self.image_height, self.image_width)}, "
                f"but received {(H, W)}."
            )

        K = self.n_filters

        # Same padding for a 3x3 filter.
        X_pad = np.pad(
            X,
            pad_width=((0, 0), (0, 0), (1, 1), (1, 1)),
            mode="constant",
        )

        # Convolution/cross-correlation.
        Z = np.zeros((B, K, H, W), dtype=np.float32)

        for r in range(3):
            for s in range(3):
                image_patch = X_pad[:, 0, r:r + H, s:s + W]
                filter_weights = self.Wc[:, 0, r, s]
                Z += image_patch[:, None, :, :] * filter_weights[None, :, None, None]

        Z += self.bc[None, :, None, None]

        # ReLU.
        A = np.maximum(Z, 0.0)

        # 2x2 max pooling.
        A_blocks = A.reshape(B, K, H // 2, 2, W // 2, 2)
        P = A_blocks.max(axis=(3, 5))

        # Dense classifier.
        F = P.reshape(B, -1)
        logits = F @ self.Wd + self.bd

        cache = (X, X_pad, Z, A, A_blocks, P, F)
        return logits, cache

    @staticmethod
    def softmax(logits):
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_values = np.exp(shifted)
        return exp_values / exp_values.sum(axis=1, keepdims=True)

    def predict_proba(self, X):
        logits, _ = self.forward(X)
        return self.softmax(logits)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def loss_and_accuracy(self, X, y):
        logits, _ = self.forward(X)
        probs = self.softmax(logits)

        loss = -np.log(probs[np.arange(len(y)), y] + 1e-12).mean()
        predictions = probs.argmax(axis=1)
        accuracy = np.mean(predictions == y)

        return float(loss), float(accuracy)

    def train_step(self, X_batch, y_batch, lr=0.05, momentum=0.9, weight_decay=1e-4):
        B = X_batch.shape[0]
        H = self.image_height
        W = self.image_width
        K = self.n_filters

        logits, cache = self.forward(X_batch)
        X, X_pad, Z, A, A_blocks, P, F = cache

        probs = self.softmax(logits)

        loss = -np.log(probs[np.arange(B), y_batch] + 1e-12).mean()

        # Gradient of cross entropy with softmax.
        dlogits = probs.copy()
        dlogits[np.arange(B), y_batch] -= 1.0
        dlogits /= B

        # Dense layer gradients.
        dWd = F.T @ dlogits + weight_decay * self.Wd
        dbd = dlogits.sum(axis=0)
        dF = dlogits @ self.Wd.T

        # Backpropagate through max pooling.
        dP = dF.reshape(B, K, H // 2, W // 2)

        max_values = P[:, :, :, None, :, None]
        pool_mask = (A_blocks == max_values)

        # If several entries are tied maxima, distribute the gradient equally.
        counts = pool_mask.sum(axis=(3, 5), keepdims=True)
        dA_blocks = pool_mask * (dP[:, :, :, None, :, None] / counts)
        dA = dA_blocks.reshape(B, K, H, W)

        # Backpropagate through ReLU.
        dZ = dA * (Z > 0)

        # Convolution gradients.
        dWc = np.zeros_like(self.Wc)
        dbc = dZ.sum(axis=(0, 2, 3))

        for r in range(3):
            for s in range(3):
                image_patch = X_pad[:, 0, r:r + H, s:s + W]
                dWc[:, 0, r, s] = (
                    dZ * image_patch[:, None, :, :]
                ).sum(axis=(0, 2, 3))

        dWc += weight_decay * self.Wc

        gradients = {
            "Wc": dWc,
            "bc": dbc,
            "Wd": dWd,
            "bd": dbd,
        }

        # Momentum SGD update.
        for name, param in self.parameters().items():
            self.velocity[name] = momentum * self.velocity[name] - lr * gradients[name]
            param += self.velocity[name]

        predictions = probs.argmax(axis=1)
        accuracy = np.mean(predictions == y_batch)

        return float(loss), float(accuracy)

## 6. Train the CNN

We now train the CNN. Because the dataset is small, this should run quickly on a normal laptop.

The printed accuracy should confirm that the model has learned a useful classifier before we apply the distortion.

In [ ]:
image_shape = X_train.shape[-2:]

model = TinyNumPyCNN(
    image_shape=image_shape,
    n_filters=8,
    n_classes=10,
    seed=42,
)

num_epochs = 30
batch_size = 64

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(1, num_epochs + 1):
    permutation = np.random.permutation(len(X_train))

    for start in range(0, len(X_train), batch_size):
        batch_indices = permutation[start:start + batch_size]

        model.train_step(
            X_train[batch_indices],
            y_train[batch_indices],
            lr=0.05,
            momentum=0.9,
            weight_decay=1e-4,
        )

    train_loss, train_acc = model.loss_and_accuracy(X_train, y_train)
    val_loss, val_acc = model.loss_and_accuracy(X_val, y_val)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"train loss = {train_loss:.4f}, train acc = {train_acc:.4f} | "
            f"val loss = {val_loss:.4f}, val acc = {val_acc:.4f}"
        )

test_loss, test_acc = model.loss_and_accuracy(X_test, y_test)

print()
print(f"Final clean test loss     : {test_loss:.4f}")
print(f"Final clean test accuracy : {test_acc:.4f}")

In [ ]:
epochs = np.arange(1, num_epochs + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs, history["train_acc"], marker="o", label="training accuracy")
plt.plot(epochs, history["val_acc"], marker="o", label="validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training progress")
plt.legend()
plt.grid(True)
plt.show()

## 7. Derivation of the discrete Laplacian

The distortion requested in the exercise is

$$
\widehat A = A + \nabla^2 A.
$$

For a smooth function of two variables,

$$
\nabla^2 A
=
\frac{\partial^2 A}{\partial x^2}
+
\frac{\partial^2 A}{\partial y^2}.
$$

For an image, we need a finite-difference version.

In one dimension, Taylor expansion gives

$$
f(x+h)
=
f(x)+h f'(x)+\frac{h^2}{2}f''(x)+O(h^3),
$$

and

$$
f(x-h)
=
f(x)-h f'(x)+\frac{h^2}{2}f''(x)+O(h^3).
$$

Adding these two equations gives

$$
f(x+h)+f(x-h)
=
2f(x)+h^2 f''(x)+O(h^4).
$$

Therefore,

$$
f''(x)
\approx
\frac{f(x+h)-2f(x)+f(x-h)}{h^2}.
$$

For image pixels we take the grid spacing to be $$h=1$$. Hence the vertical second derivative is approximated by

$$
\frac{\partial^2 A}{\partial x^2}
\approx
A_{i+1,j}-2A_{i,j}+A_{i-1,j},
$$

and the horizontal second derivative is approximated by

$$
\frac{\partial^2 A}{\partial y^2}
\approx
A_{i,j+1}-2A_{i,j}+A_{i,j-1}.
$$

Adding the two directions gives the five-point discrete Laplacian

$$
(\nabla^2 A)_{i,j}
=
A_{i+1,j}
+
A_{i-1,j}
+
A_{i,j+1}
+
A_{i,j-1}
-
4A_{i,j}.
$$

## 8. Laplacian as a convolution kernel

The same operation can be written as filtering the image with the kernel

$$
L=
\begin{pmatrix}
0 & 1 & 0\\
1 & -4 & 1\\
0 & 1 & 0
\end{pmatrix}.
$$

This is the key conceptual connection for the tutorial: the CNN learns convolutional filters, while the distortion applies a fixed convolutional filter.

In [ ]:
laplacian_kernel = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0],
], dtype=np.float32)

print("Laplacian kernel:")
print(laplacian_kernel)


def image_laplacian(images):
    """Compute the five-point discrete Laplacian."""
    single_image = (images.ndim == 2)

    if single_image:
        images = images[np.newaxis, np.newaxis, :, :]
    elif images.ndim == 3:
        images = images[:, np.newaxis, :, :]

    N, C, H, W = images.shape

    padded = np.pad(
        images,
        pad_width=((0, 0), (0, 0), (1, 1), (1, 1)),
        mode="constant",
    )

    lap = (
        padded[:, :, 0:H,   1:W+1] +
        padded[:, :, 2:H+2, 1:W+1] +
        padded[:, :, 1:H+1, 0:W] +
        padded[:, :, 1:H+1, 2:W+2] -
        4.0 * padded[:, :, 1:H+1, 1:W+1]
    )

    if single_image:
        return lap[0, 0]

    return lap


def laplacian_attack(images, epsilon=1.0, clip=True):
    """Apply A_hat = A + epsilon * Laplacian(A)."""
    attacked = images + epsilon * image_laplacian(images)

    if clip:
        attacked = np.clip(attacked, 0.0, 1.0)

    return attacked

## 9. Why clipping is used

The mathematical distortion is

$$
\widehat A = A + \nabla^2 A.
$$

The raw array $$A+\nabla^2A$$ may contain values below 0 or above 1. The network, however, was trained on images whose pixel values lie in the interval

$$
0 \leq A_{i,j} \leq 1.
$$

Therefore, for the actual network input we use the clipped image

$$
\widehat A
=
\operatorname{clip}\!\left(A+\nabla^2A,0,1\right).
$$

This preserves the intended distortion while producing a valid grayscale image for the trained classifier.

## 10. Visualize one successful attack

We choose a test image that satisfies two conditions:

1. it is classified correctly before the distortion,
2. it is classified incorrectly after the distortion with $$\varepsilon=1$$.

This gives a clear demonstration of the effect

$$
A
\longrightarrow
\widehat A = A + \nabla^2 A.
$$

In [ ]:
clean_predictions = model.predict(X_test)
attacked_predictions_eps1 = model.predict(laplacian_attack(X_test, epsilon=1.0))

candidate_indices = np.where(
    (clean_predictions == y_test) &
    (attacked_predictions_eps1 != y_test)
)[0]

print("Number of clean-correct but attack-wrong examples:", len(candidate_indices))

if len(candidate_indices) == 0:
    example_index = 0
    print("No clean-correct/attack-wrong example was found; using index 0.")
else:
    example_index = candidate_indices[0]

A = X_test[example_index:example_index+1]
A_single = A[0, 0]

lap_A = image_laplacian(A)[0, 0]
A_tilde = A + image_laplacian(A)
A_hat = laplacian_attack(A, epsilon=1.0)

probs_A = model.predict_proba(A)[0]
probs_hat = model.predict_proba(A_hat)[0]

pred_A = probs_A.argmax()
pred_hat = probs_hat.argmax()

print("Selected test index:", example_index)
print("True label:", y_test[example_index])
print("Prediction on clean A:", pred_A, "confidence:", probs_A[pred_A])
print("Prediction on attacked A_hat:", pred_hat, "confidence:", probs_hat[pred_hat])
print("Raw A + Laplacian(A) value range:", A_tilde.min(), "to", A_tilde.max())
print("Clipped A_hat value range:", A_hat.min(), "to", A_hat.max())

In [ ]:
plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(A_single, cmap="gray", vmin=0, vmax=1)
plt.title(f"Original A\npred = {pred_A}")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(lap_A, cmap="gray")
plt.title(r"Laplacian $\nabla^2 A$")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(A_tilde[0, 0], cmap="gray")
plt.title(r"Raw $A+\nabla^2A$")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(A_hat[0, 0], cmap="gray", vmin=0, vmax=1)
plt.title(r"Clipped $\widehat A$" + f"\npred = {pred_hat}")
plt.axis("off")

plt.suptitle("Laplacian attack on one test image")
plt.tight_layout()
plt.show()

## 11. Attack strength

The requested distortion is

$$
\widehat A = A + \nabla^2 A.
$$

For analysis, it is useful to introduce a distortion-strength parameter $$\varepsilon$$:

$$
\widehat A_{\varepsilon}
=
\operatorname{clip}\!\left(
A+\varepsilon \nabla^2 A,
0,1
\right).
$$

The original instruction corresponds to

$$
\varepsilon=1.
$$

Varying $$\varepsilon$$ lets us see when the classifier begins to fail.

In [ ]:
epsilons_to_show = [0.0, 0.05, 0.10, 0.20, 0.50, 1.00]

plt.figure(figsize=(14, 3))

for k, eps in enumerate(epsilons_to_show):
    A_eps = laplacian_attack(A, epsilon=eps)
    probs_eps = model.predict_proba(A_eps)[0]
    pred_eps = probs_eps.argmax()
    conf_eps = probs_eps[pred_eps]

    plt.subplot(1, len(epsilons_to_show), k + 1)
    plt.imshow(A_eps[0, 0], cmap="gray", vmin=0, vmax=1)
    plt.title(f"eps={eps:.2f}\npred={pred_eps}, conf={conf_eps:.2f}")
    plt.axis("off")

plt.suptitle("Increasing Laplacian attack strength")
plt.tight_layout()
plt.show()

## 12. Accuracy under attack

We now evaluate the whole test set under different distortion strengths.

For each value of $$\varepsilon$$, we compute

$$
\widehat A_{\varepsilon}
=
\operatorname{clip}\!\left(
A+\varepsilon \nabla^2 A,
0,1
\right)
$$

for every image and measure the test accuracy.

In [ ]:
eps_grid = np.linspace(0.0, 1.0, 11)
attacked_accuracies = []

for eps in eps_grid:
    X_test_attacked = laplacian_attack(X_test, epsilon=float(eps))
    loss_eps, acc_eps = model.loss_and_accuracy(X_test_attacked, y_test)
    attacked_accuracies.append(acc_eps)
    print(f"epsilon = {eps:.2f} | attacked test accuracy = {acc_eps:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(eps_grid, attacked_accuracies, marker="o")
plt.xlabel(r"Attack strength $\varepsilon$")
plt.ylabel("Test accuracy")
plt.title(r"Accuracy under $\widehat A_\varepsilon=A+\varepsilon\nabla^2A$")
plt.grid(True)
plt.show()

## 13. Interpretation

The clean CNN performs well on the original test images. The Laplacian distortion modifies the local structure of the image.

This is relevant because CNNs are designed to learn local patterns such as edges, strokes, corners, and local intensity transitions.

The Laplacian is also local. It compares each pixel with its nearest neighbors:

$$
(\nabla^2 A)_{i,j}
=
A_{i+1,j}
+
A_{i-1,j}
+
A_{i,j+1}
+
A_{i,j-1}
-
4A_{i,j}.
$$

Therefore, the distortion perturbs precisely the kind of local structure that the CNN uses.

This is not the same as a standard gradient-based adversarial attack such as FGSM. FGSM uses the gradient of the loss with respect to the input image. Here, the perturbation is fixed by an image-processing operator and does not require access to the trained network.

## 14. Student tasks

1. Report the clean test accuracy of the CNN.
2. Report the attacked test accuracy for the requested distortion, corresponding to $$\varepsilon=1$$.
3. For the displayed example, compare the clean prediction and the attacked prediction.
4. Determine the smallest value of $$\varepsilon$$ in the plotted list for which the prediction changes.
5. Explain why the Laplacian is large near edges or sharp transitions.
6. Explain why this distortion is not a gradient-based adversarial attack.
7. Propose one method to improve robustness against this distortion.

## 15. Optional extension for students: repeat the experiment with MNIST

The present notebook uses the built-in `scikit-learn` digits dataset because it is small and requires no download. Students who want a larger experiment can repeat the same workflow on MNIST.

MNIST consists of 28 by 28 grayscale handwritten digit images. The same CNN class above can be reused by setting

```python
model = TinyNumPyCNN(image_shape=(28, 28), n_filters=8, n_classes=10)
```

A possible route is:

```python
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X_mnist = mnist.data.astype(np.float32) / 255.0
y_mnist = mnist.target.astype(np.int64)

X_mnist = X_mnist.reshape(-1, 1, 28, 28)

X_train, X_test, y_train, y_test = train_test_split(
    X_mnist,
    y_mnist,
    test_size=10000,
    random_state=0,
    stratify=y_mnist,
)

model = TinyNumPyCNN(image_shape=(28, 28), n_filters=8, n_classes=10)
```

Then students should repeat the same steps:

1. train the CNN,
2. evaluate clean accuracy,
3. compute the discrete Laplacian $$\nabla^2A$$,
4. form the distorted image

$$
\widehat A = A + \nabla^2 A,
$$

5. plot test accuracy as a function of $$\varepsilon$$.

For a classroom exercise, students may train on a smaller MNIST subset, for example 5000 to 10000 training images, so that the pure NumPy implementation remains reasonably fast.

## 16. Summary

This notebook implements the complete requested exercise:

$$
\text{train a simple CNN}
\quad\longrightarrow\quad
\widehat A = A + \nabla^2 A
\quad\longrightarrow\quad
\text{compare predictions and accuracy}.
$$

The main conceptual point is that both the CNN and the Laplacian distortion operate locally. The CNN learns local filters; the Laplacian applies a fixed local filter. The experiment therefore tests whether the trained local-feature classifier is stable under a structured local distortion.